# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, examine, and analyze the FAIR^2 dataset using the [mlcroissant](https://pypi.org/project/mlcroissant/) Python library.

## Dataset Source
Croissant JSON-LD Schema URL:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Make sure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Data Title: {metadata.name}\n")
print(f"Data Description: {metadata.description}\n")
print(f"Authors: {[getattr(a, '@id', a) for a in getattr(metadata, 'author', [])]}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}\n")


## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate the available RecordSets in the dataset and preview their fields and columns.

**Note**: Each entity is referenced via its `@id`, following Croissant and FAIR^2 best practices.

In [ ]:
# List all record sets with @id and name
record_sets = getattr(metadata, 'recordSet', [])
if isinstance(record_sets, dict):
    record_sets = [record_sets]
elif not isinstance(record_sets, list):
    record_sets = []

if not record_sets:
    print("No record sets found in this dataset. Please inspect the metadata for schema details.")
else:
    print(f"{len(record_sets)} record set(s) found in the dataset.")
    for i, rs in enumerate(record_sets):
        rs_id = getattr(rs, '@id', None) or (rs.get('@id') if isinstance(rs, dict) else None)
        rs_name = getattr(rs, 'name', None) or (rs.get('name') if isinstance(rs, dict) else None)
        print(f"[{i + 1}] RecordSet @id: {rs_id}, name: {rs_name}")
        # Inspect fields
        fields = getattr(rs, 'field', []) or (rs.get('field') if isinstance(rs, dict) else []) or []
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            f_id = getattr(f, '@id', None) or (f.get('@id') if isinstance(f, dict) else None)
            f_name = getattr(f, 'name', None) or (f.get('name') if isinstance(f, dict) else None)
            print(f"    - Field @id: {f_id}, name: {f_name}")
        # Inspect columns
        columns = getattr(rs, 'column', []) or (rs.get('column') if isinstance(rs, dict) else []) or []
        if isinstance(columns, dict):
            columns = [columns]
        for c in columns:
            c_id = getattr(c, '@id', None) or (c.get('@id') if isinstance(c, dict) else None)
            c_name = getattr(c, 'name', None) or (c.get('name') if isinstance(c, dict) else None)
            print(f"    - Column @id: {c_id}, name: {c_name}")

    print("\nYou will need these @id values for the following extraction steps.")

# For demonstration, print out metadata
print("\n--- Metadata keys ---")
for k in dir(metadata):
    if not k.startswith('_') and not callable(getattr(metadata, k)):
        print(f"{k}: {getattr(metadata, k)}")

## 3. Data Extraction
Load tabular data from a selected record set. All subsequent references use `@id` fields for precise, reproducible selection.

If the schema contains multiple record sets, you may specify those below. We'll demonstrate with the first available RecordSet (if present).

In [ ]:
# If no record sets are present, print a message, else extract records for each RecordSet
dataframes = {}
if not record_sets:
    print("No tabular record sets available for extraction in this dataset.")
else:
    record_set_ids = [rs.get('@id', getattr(rs, '@id', None)) for rs in record_sets]
    print("Extracting records from RecordSets by @id:")
    for rset_id in record_set_ids:
        print(f" - {rset_id}")
        records_iter = dataset.records(record_set=rset_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[rset_id] = df
            print(f"Loaded shape for {rset_id}: {df.shape}")
            print("Columns:", df.columns.tolist())
            display(df.head())
        else:
            print(f"! No records found for RecordSet {rset_id}")
    # For convenience, assign the first RecordSet for further analysis
    main_rs_id = record_set_ids[0] if record_set_ids else None


## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering, normalizing, and grouping.

You can adjust the numeric field (`numeric_field_id`) and grouping field (`group_field_id`) from those listed in the previous Data Extraction block.

In [ ]:
# Example EDA on a main record set, if available
if 'main_rs_id' not in locals() or main_rs_id not in dataframes:
    print('No tabular data available for EDA.')
else:
    df = dataframes[main_rs_id]
    print(f"Shape: {df.shape}\nColumns: {df.columns.tolist()}")
    # Try guessing a numeric field (e.g. MW or abundance column)
    import numpy as np
    import re

    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
    if not numeric_candidates:
        # Try to cast columns with numeric names to float
        for col in df.columns:
            # Heuristics: MW, coverage, abundance, peptide_count etc
            if re.search(r'(mw|weight|count|abundance|coverage|peptide|normalized|value|score)', col, flags=re.I):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except:
                    continue
        numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]

    if not numeric_candidates:
        print('No numeric field found for EDA demo.')
    else:
        numeric_field = numeric_candidates[0]
        print(f"Chosen numeric field (by heuristics): {numeric_field}")
        # Show its description if available in metadata
        # Filtering
        threshold = np.nanmedian(df[numeric_field])
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered (>{threshold}) records: {len(filtered_df)} out of {len(df)}")

        # Normalization
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[f'{numeric_field}_normalized'] = (filtered_df[numeric_field] - mean) / std
        print(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

        # Try grouping by another field
        # For proteins may group by 'Description' or 'accession', pick heuristically
        group_candidates = [col for col in df.columns if col.lower() not in [numeric_field.lower()] and df[col].dtype == 'O']
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field} (top5):")
            print(grouped_df.head())

## 5. Visualization
Plot distributions and field relationships to aid exploration.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'main_rs_id' in locals() and main_rs_id in dataframes:
    df = dataframes[main_rs_id]

    if 'numeric_field' in locals():
        plt.figure(figsize=(8,3))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='skyblue')
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()

        # If a group field exists, show boxplot
        if 'group_field' in locals() and group_field:
            plt.figure(figsize=(10,4))
            order = df[group_field].value_counts().index[:8]
            sns.boxplot(data=df, x=group_field, y=numeric_field, order=order)
            plt.title(f"{numeric_field} by {group_field} (Top 8 groups)")
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, process, and visualize a Croissant-encoded dataset using the `mlcroissant` library. All entities (record sets, fields, columns) are referenced via their `@id` as per the FAIR^2 and Croissant best practices.

- For production analyses, follow the same structure, but reference entity `@id`s explicitly for reproducibility.
- See the dataset documentation or inspect its metadata for further details on field semantics and study design.

**Key findings and further steps:**
- Key quantitative fields, record set structure, and summary statistics can be retrieved programmatically.
- For domain-specific analysis, expand grouping, filtering, and plotting to focus on biological or experimental variables.
- Please visit [mlcroissant documentation](https://mlcommons.github.io/croissant/python) for advanced usage.